# 1. Inicialización del Entorno Spark

**Dimensión 3: Análisis y Modelado de Duración de Viajes** — Regresión de `duration_minutes`.

Integrante: **Jose Miguel Condo Huamani** — Data Geniuses

Procesamiento batch distribuido sobre los viajes de Citi Bike NYC.


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("U1_Duracion_Viajes_CitiBike")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/11 14:23:50 WARN Utils: Your hostname, Jhack, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/11 14:23:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/11 14:23:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


# 2. Definición del Esquema Explícito y Carga de Datos (Capa Bronze)

Lectura raw con `StructType` explícito. La ruta de datos es portable: en el
contenedor usa `/opt/UNIDAD1/data/*.csv`; en local `../data/*.csv`.


In [2]:
import os as _os

_has_opt = _os.path.exists("/opt/UNIDAD1")
_cwd = _os.getcwd()

DATA_PATH = "/opt/UNIDAD1/data/*.csv"
GOLD_ROOT = "/opt/UNIDAD1/gold"
if not _has_opt:
    DATA_PATH = _os.path.join(_cwd, "..", "data", "*.csv")
    GOLD_ROOT = _os.path.join(_cwd, "..", "gold")

from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, DoubleType,
)

schema_citibike = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True),
    StructField("start_station_id", StringType(), True),
    StructField("end_station_name", StringType(), True),
    StructField("end_station_id", StringType(), True),
    StructField("start_lat", DoubleType(), True),
    StructField("start_lng", DoubleType(), True),
    StructField("end_lat", DoubleType(), True),
    StructField("end_lng", DoubleType(), True),
    StructField("member_casual", StringType(), True),
])

df = spark.read.schema(schema_citibike).option("header", True).csv(DATA_PATH)

26/09/11 14:23:54 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/Dim-Duracion-de-viajes/../data/*.csv.
java.io.FileNotFoundException: File /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/data/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark

# 2.1 Verificación Inicial de Registros


In [3]:
total_registros = df.count()
print(f"Registros ingeridos: {total_registros:,}")

Registros ingeridos: 4,993,137


In [4]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- member_casual: string (nullable = true)



# 3. Capa Silver: Cálculo de Duración y Filtro de Validez

`duration_minutes = (ended_at - started_at) / 60.0`, reteniendo viajes con
duración operativa entre **1 y 180 minutos** y coordenadas completas.


In [5]:
from pyspark.sql.functions import col, unix_timestamp, sqrt, pow

df = df.withColumn(
    "duration_minutes",
    (unix_timestamp(col("ended_at")) - unix_timestamp(col("started_at"))) / 60.0,
).withColumn(
    "approx_distance_km",
    sqrt(pow((col("end_lat") - col("start_lat")) * 111.0, 2)
         + pow((col("end_lng") - col("start_lng")) * 85.0, 2)),
)

In [6]:
df_clean = df.filter(
    col("duration_minutes").between(1.0, 180.0)
    & col("start_lat").isNotNull()
    & col("end_lat").isNotNull()
    & col("rideable_type").isNotNull()
    & col("member_casual").isNotNull()
)

n_validos = df_clean.count()
print(f"Viajes válidos: {n_validos:,} | Descartados: {total_registros - n_validos:,}")

Viajes válidos: 4,971,847 | Descartados: 21,290


In [7]:
df_clean.select(
    "ride_id", "started_at", "ended_at", "duration_minutes",
    "approx_distance_km", "rideable_type", "member_casual",
).show(5, truncate=False)

+----------------+-----------------------+-----------------------+------------------+-------------------+-------------+-------------+
|ride_id         |started_at             |ended_at               |duration_minutes  |approx_distance_km |rideable_type|member_casual|
+----------------+-----------------------+-----------------------+------------------+-------------------+-------------+-------------+
|FB2708B4615FC7B1|2026-07-04 15:52:29.356|2026-07-04 16:03:08.021|10.65             |2.1194169058781442 |electric_bike|member       |
|6D9C1D2C900149A3|2026-07-09 00:31:21.213|2026-07-09 00:35:10.506|3.816666666666667 |0.43855034866229575|classic_bike |member       |
|C945BA79D67B1EAE|2026-07-02 22:51:30.134|2026-07-02 22:56:13.056|4.716666666666667 |0.3776322710625763 |electric_bike|member       |
|9896AB3B453BED3B|2026-07-01 15:50:55.534|2026-07-01 15:58:18.824|7.383333333333334 |1.407450415396838  |electric_bike|member       |
|ACADC9E7E131ECBC|2026-07-08 06:13:41.033|2026-07-08 06:33:30.

# 4. Feature Engineering: Variables Temporales y Espaciales

Extraemos las variables explicativas: `start_hour`, `day_of_week`, `is_weekend`
y la distancia euclidiana aproximada (coordenadas → km).


In [8]:
from pyspark.sql.functions import col as ccol, to_date, hour, dayofweek, when, month

df_feat = (
    df_clean.withColumn("trip_date", to_date(ccol("started_at")))
            .withColumn("start_hour", hour(ccol("started_at")))
            .withColumn("day_of_week", dayofweek(ccol("started_at")))
            .withColumn("start_month", month(ccol("started_at")))
            .withColumn(
                "is_weekend",
                when(ccol("day_of_week").isin(1, 7), 1).otherwise(0),
            )
)

In [9]:
print("Distribución de la variable objetivo (duration_minutes):")
df_feat.describe("duration_minutes", "approx_distance_km", "start_hour").show()

Distribución de la variable objetivo (duration_minutes):


+-------+------------------+------------------+------------------+
|summary|  duration_minutes|approx_distance_km|        start_hour|
+-------+------------------+------------------+------------------+
|  count|           4971847|           4971847|           4971847|
|   mean|12.897422483703329|2.0863094302280856|14.183285809076587|
| stddev| 11.92820328871835|1.7667685496888585|  5.29008069336773|
|    min|1.0166666666666666|               0.0|                 0|
|    max|179.98333333333332|28.649013774550358|                23|
+-------+------------------+------------------+------------------+



In [10]:
df_feat.select(
    "start_hour", "day_of_week", "is_weekend",
    "rideable_type", "member_casual", "approx_distance_km",
).show(5, truncate=False)

+----------+-----------+----------+-------------+-------------+-------------------+
|start_hour|day_of_week|is_weekend|rideable_type|member_casual|approx_distance_km |
+----------+-----------+----------+-------------+-------------+-------------------+
|15        |7          |1         |electric_bike|member       |2.1194169058781442 |
|0         |5          |0         |classic_bike |member       |0.43855034866229575|
|22        |5          |0         |electric_bike|member       |0.3776322710625763 |
|15        |4          |0         |electric_bike|member       |1.407450415396838  |
|6         |4          |0         |classic_bike |member       |3.5854709733140906 |
+----------+-----------+----------+-------------+-------------+-------------------+
only showing top 5 rows


# 5. Preparación ML: Codificación, VectorAssembler y Split

- `StringIndexer` + `OneHotEncoder` para `member_casual` y `rideable_type`.
- `VectorAssembler` empaqueta temporales + distancia + categóricas en `features`.
- División Train (80%) / Test (20%) sin *data leakage* (ajuste sobre Train).


In [11]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

categoricas = ["member_casual", "rideable_type"]
numericas = ["start_hour", "day_of_week", "start_month", "is_weekend", "approx_distance_km"]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx").setHandleInvalid("keep")
    for c in categoricas
]
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in categoricas],
    outputCols=[c + "_ohe" for c in categoricas],
)
assembler = VectorAssembler(
    inputCols=numericas + [c + "_ohe" for c in categoricas],
    outputCol="features",
)

In [12]:
train_raw, test_raw = df_feat.randomSplit([0.8, 0.2], seed=42)

pipeline_prep = Pipeline(stages=indexers + [encoder, assembler])
prep_model = pipeline_prep.fit(train_raw)

train = prep_model.transform(train_raw)
test = prep_model.transform(test_raw)
print(f"Train: {train.count():,} | Test: {test.count():,}")

Train: 3,978,452 | Test: 993,395


# 6. Entrenamiento de Modelos de Regresión (Spark MLlib)

Se entrenan **LinearRegression** y **RandomForestRegressor** con
`labelCol="duration_minutes"`, seleccionando el de menor RMSE en test.


In [13]:
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

lr = LinearRegression(featuresCol="features", labelCol="duration_minutes",
                      regParam=0.05, maxIter=30)
model_lr = lr.fit(train)

rf = RandomForestRegressor(featuresCol="features", labelCol="duration_minutes",
                           numTrees=40, maxDepth=10, seed=42)
model_rf = rf.fit(train)

preds_lr = model_lr.transform(test)
preds_rf = model_rf.transform(test)
print("Modelos de regresión entrenados correctamente.")

26/09/11 14:24:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


26/09/11 14:24:53 WARN MemoryStore: Not enough space to cache rdd_117_3 in memory! (computed 41.7 MiB so far)
26/09/11 14:24:53 WARN BlockManager: Persisting block rdd_117_3 to disk instead.
26/09/11 14:24:53 WARN MemoryStore: Not enough space to cache rdd_117_9 in memory! (computed 41.7 MiB so far)


26/09/11 14:24:53 WARN BlockManager: Persisting block rdd_117_9 to disk instead.
26/09/11 14:24:53 WARN MemoryStore: Not enough space to cache rdd_117_7 in memory! (computed 41.7 MiB so far)
26/09/11 14:24:53 WARN BlockManager: Persisting block rdd_117_7 to disk instead.
26/09/11 14:24:53 WARN MemoryStore: Not enough space to cache rdd_117_2 in memory! (computed 41.7 MiB so far)
26/09/11 14:24:53 WARN BlockManager: Persisting block rdd_117_2 to disk instead.
26/09/11 14:24:53 WARN MemoryStore: Not enough space to cache rdd_117_10 in memory! (computed 62.6 MiB so far)
26/09/11 14:24:53 WARN BlockManager: Persisting block rdd_117_10 to disk instead.
26/09/11 14:24:53 WARN MemoryStore: Not enough space to cache rdd_117_8 in memory! (computed 41.7 MiB so far)
26/09/11 14:24:53 WARN BlockManager: Persisting block rdd_117_8 to disk instead.
26/09/11 14:24:53 WARN MemoryStore: Not enough space to cache rdd_117_11 in memory! (computed 62.6 MiB so far)
26/09/11 14:24:53 WARN BlockManager: Persi

26/09/11 14:24:56 WARN MemoryStore: Not enough space to cache rdd_117_5 in memory! (computed 93.8 MiB so far)
26/09/11 14:24:56 WARN BlockManager: Persisting block rdd_117_5 to disk instead.
26/09/11 14:24:56 WARN MemoryStore: Not enough space to cache rdd_117_0 in memory! (computed 93.8 MiB so far)
26/09/11 14:24:56 WARN BlockManager: Persisting block rdd_117_0 to disk instead.


26/09/11 14:24:57 WARN MemoryStore: Not enough space to cache rdd_117_1 in memory! (computed 93.8 MiB so far)
26/09/11 14:24:57 WARN BlockManager: Persisting block rdd_117_1 to disk instead.
26/09/11 14:24:57 WARN MemoryStore: Not enough space to cache rdd_117_6 in memory! (computed 93.8 MiB so far)
26/09/11 14:24:57 WARN BlockManager: Persisting block rdd_117_6 to disk instead.


26/09/11 14:24:59 WARN MemoryStore: Not enough space to cache rdd_117_3 in memory! (computed 62.6 MiB so far)
26/09/11 14:24:59 WARN MemoryStore: Not enough space to cache rdd_117_9 in memory! (computed 62.6 MiB so far)
26/09/11 14:24:59 WARN MemoryStore: Not enough space to cache rdd_117_8 in memory! (computed 62.6 MiB so far)
26/09/11 14:24:59 WARN MemoryStore: Not enough space to cache rdd_117_7 in memory! (computed 62.6 MiB so far)


... (18 outputs left out)

26/09/11 14:25:47 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB


26/09/11 14:25:48 WARN MemoryStore: Not enough space to cache rdd_117_9 in memory! (computed 62.6 MiB so far)
26/09/11 14:25:48 WARN MemoryStore: Not enough space to cache rdd_117_2 in memory! (computed 62.6 MiB so far)
26/09/11 14:25:48 WARN MemoryStore: Not enough space to cache rdd_117_7 in memory! (computed 62.6 MiB so far)


26/09/11 14:25:48 WARN MemoryStore: Not enough space to cache rdd_117_8 in memory! (computed 93.8 MiB so far)
26/09/11 14:25:48 WARN MemoryStore: Not enough space to cache rdd_117_3 in memory! (computed 93.8 MiB so far)


Modelos de regresión entrenados correctamente.


# 7. Evaluación Formal del Modelo (Métricas de Calidad)

`RegressionEvaluator`: **RMSE**, **MAE** y **R²** para ambos modelos.


In [14]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse_eval = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol="duration_minutes", predictionCol="prediction", metricName="r2")

resultado = []
for nombre, preds in [("LinearRegression", preds_lr), ("RandomForestRegressor", preds_rf)]:
    resultado.append((
        nombre,
        round(rmse_eval.evaluate(preds), 4),
        round(mae_eval.evaluate(preds), 4),
        round(r2_eval.evaluate(preds), 4),
    ))

df_eval = spark.createDataFrame(
    resultado,
    ["Modelo", "RMSE (min)", "MAE (min)", "R2"],
)
df_eval.show(truncate=False)

mejor_idx = min(range(len(resultado)), key=lambda i: resultado[i][1])
mejor_modelo, mejor_preds = (model_rf, preds_rf) if resultado[mejor_idx][0] == "RandomForestRegressor" else (model_lr, preds_lr)
print(f">> Modelo seleccionado por menor RMSE: {resultado[mejor_idx][0]}")

+---------------------+----------+---------+------+
|Modelo               |RMSE (min)|MAE (min)|R2    |
+---------------------+----------+---------+------+
|LinearRegression     |9.2532    |4.6383   |0.3962|
|RandomForestRegressor|9.0613    |4.6707   |0.4209|
+---------------------+----------+---------+------+

>> Modelo seleccionado por menor RMSE: RandomForestRegressor


In [15]:
print("Muestra de predicciones (duración real vs estimada):")
mejor_preds.select("duration_minutes", "prediction").show(10, truncate=False)

Muestra de predicciones (duración real vs estimada):


+------------------+------------------+
|duration_minutes  |prediction        |
+------------------+------------------+
|14.0              |18.569491086125918|
|3.3333333333333335|6.806309431158679 |
|2.783333333333333 |8.139167574327061 |
|1.75              |5.725268457518122 |
|1.9666666666666666|8.406367238000536 |
|7.233333333333333 |11.841028158066152|
|27.05             |12.402632260830798|
|8.8               |10.193689121214138|
|6.983333333333333 |8.397101456844954 |
|9.233333333333333 |9.268231691273474 |
+------------------+------------------+
only showing top 10 rows


# 8. Persistencia Analítica en Capa Gold

Guardamos las predicciones de duración en **Parquet**.


In [16]:
from pyspark.sql.functions import round as spark_round

pred_final = (
    mejor_preds.select(
        ccol("trip_date"), ccol("start_hour"), ccol("day_of_week"),
        ccol("is_weekend"), ccol("member_casual"), ccol("rideable_type"),
        ccol("approx_distance_km"),
        spark_round(ccol("duration_minutes"), 2).alias("duration_minutes"),
        spark_round(ccol("prediction"), 2).alias("prediction"),
    )
)

GOLD_PATH = GOLD_ROOT + "/duracion_predictions.parquet"
pred_final.write.mode("overwrite").parquet(GOLD_PATH)
print(f"Predicciones persistidas: {GOLD_PATH}")

Predicciones persistidas: /home/jhackns/bigdata/Proyectos/grupo-x-citibike-intelligent-mobility/pyspark/jupyter/UNIDAD1/Dim-Duracion-de-viajes/../gold/duracion_predictions.parquet


In [17]:
gold_check = spark.read.parquet(GOLD_PATH)
print(f"Filas en capa Gold: {gold_check.count():,}")
gold_check.show(5, truncate=False)

Filas en capa Gold: 993,395
+----------+----------+-----------+----------+-------------+-------------+------------------+----------------+----------+
|trip_date |start_hour|day_of_week|is_weekend|member_casual|rideable_type|approx_distance_km|duration_minutes|prediction|
+----------+----------+-----------+----------+-------------+-------------+------------------+----------------+----------+
|2026-07-14|18        |3          |0         |casual       |electric_bike|3.164932905694518 |14.0            |18.57     |
|2026-07-08|19        |4          |0         |member       |electric_bike|0.4526637597588953|3.33            |6.81      |
|2026-07-11|22        |7          |1         |member       |electric_bike|0.5153110541218243|2.78            |8.14      |
|2026-07-07|6         |3          |0         |member       |electric_bike|0.3732847162682418|1.75            |5.73      |
|2026-07-05|19        |1          |1         |member       |electric_bike|0.5299097601577566|1.97            |8.41    